# ActionShap — Review-3 Remaining Tasks

End-to-end notebook for the experiments that could not be run in the review
sandbox (datasets are not downloadable there). Run this notebook on a machine
with the data; it executes every remaining mandatory revision from the third
review, writes JSON results, and assembles the manuscript tables.

| Section | Review issue addressed | Expected runtime (CPU) |
|---|---|---|
| 1-2 | exact-Shapley validation; interactions; bounded/continuous LIME; finite differences; integrated gradients; prospective audit; ablations; runtime/memory (crit. 1,2,5; high 4,9,11) | 20-40 min each |
| 3 | competitive sequential recommender with inference-time weighting (crit. 3) | 1-2 h |
| 4 | enlarged full-catalogue cohort, active-n stability (high 12) | 30-60 min |
| 5 | aggregation into Table D4 + runtime table | seconds |

Outputs: `results/review3/*.json`, `tables/review3_tableD4.tex`,
`results/review3/summary.md`. When finished, commit and push this branch; the
manuscript integration (Table D4, §6 estimator validation, §6.6 runtimes) will
be completed from those artifacts.

Prerequisites: `pip install -r requirements-recommendation.txt` and, for §3,
`pip install torch`. See `docs/REVIEW3_EXPERIMENT_GUIDE.md` for the CLI
equivalents of every cell.

In [1]:
import os, subprocess, sys
from pathlib import Path
CODE = Path(os.getcwd())
if not (CODE / "scripts" / "run_review3_experiments.py").exists():
    CODE = Path("paper-ideas/ActionShap/code")  # opened from repo root
os.chdir(CODE)
print("python:", sys.version.split()[0])
import numpy, scipy, pandas, sklearn
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__)
try:
    import torch
    print("torch", torch.__version__, "(SASRec enabled)")
except Exception:
    print("torch NOT installed - section 3 will be skipped")

python: 3.12.13
numpy 2.4.6 | scipy 1.17.1 | pandas 2.3.3
torch 2.3.1 (SASRec enabled)


In [2]:
# sanity: full test suite (synthetic-data checks for the new modules)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "-x"], check=False)

........................................................................ [ 61%]
.............................................                            [100%]
=============================== warnings summary ===============================
tests/test_recommendation_protocol.py::test_method_summary_counts_users_missing_under_every_seed
  /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/code/scripts/make_paper_assets.py:1184: SyntaxWarning: invalid escape sequence '\%'
    lines.append(f"{label} & {labels[method]} & {_tex_number(effect['mean'])} & {_tex_number(100 * success['mean'], 1)}\% \\\\")

tests/test_recommendation_protocol.py::test_method_summary_counts_users_missing_under_every_seed
  /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/code/scripts/make_paper_assets.py:1255: SyntaxWarning: invalid escape sequence '\%'
    f"{_tex_number(100 * row.rank_valid_fraction, 1)}\% & "

tests/test_recommendation_protocol.py::test_method_summary_cou

CompletedProcess(args=['/Users/mlouhichi/Desktop/actionShap/.venv/bin/python', '-m', 'pytest', 'tests/', '-q', '-x'], returncode=0)

## 0. Datasets

MovieLens-1M (GroupLens) and Amazon Digital Music (UCSD 2018, 5-core, rating>=4).
Set `ACCEPT_TERMS=True` only after reviewing the maintainers' terms; existing
valid files are reused.

In [3]:
ACCEPT_TERMS = False
ml_ok = Path("data/ml-1m/ratings.dat").exists()
am_ok = Path("data/amazon-digital-music/interactions.csv").exists()
if (not ml_ok or not am_ok) and ACCEPT_TERMS:
    subprocess.run([sys.executable, "scripts/download_datasets.py",
                    "--dataset", "all", "--accept-dataset-terms"], check=True)
elif not (ml_ok and am_ok):
    raise SystemExit("datasets missing: set ACCEPT_TERMS=True after reviewing terms, "
                     "or place files under data/ (see scripts/download_datasets.py)")
print("MovieLens:", ml_ok, "| Amazon:", am_ok)

MovieLens: True | Amazon: True


## 1. ItemKNN — MovieLens-1M

Per user: MC Shapley vs exact Shapley (n_u<=12 subset), pair interactions and
additive-vs-realized B=2 comparison, bounded binary/continuous LIME, finite
differences, integrated gradients, prospective top-1 audit, ablations
(forced / magnitude-only / interaction-aware).

In [4]:
subprocess.run([sys.executable, "scripts/run_review3_experiments.py",
                "--dataset", "movielens", "--users", "250",
                "--exact-users", "100", "--exact-max", "12"], check=True)

[review3] 25/250 users done
[review3] 50/250 users done
[review3] 75/250 users done
[review3] 100/250 users done
[review3] 125/250 users done
[review3] 150/250 users done
[review3] 175/250 users done
[review3] 200/250 users done
[review3] 225/250 users done
[review3] 250/250 users done
wrote results/review3/review3_movielens_itemknn.json


CompletedProcess(args=['/Users/mlouhichi/Desktop/actionShap/.venv/bin/python', 'scripts/run_review3_experiments.py', '--dataset', 'movielens', '--users', '250', '--exact-users', '100', '--exact-max', '12'], returncode=0)

## 2. ItemKNN — Amazon Digital Music

In [5]:
subprocess.run([sys.executable, "scripts/run_review3_experiments.py",
                "--dataset", "amazon", "--users", "250",
                "--exact-users", "100", "--exact-max", "12"], check=True)

[review3] 25/250 users done
[review3] 50/250 users done
[review3] 75/250 users done
[review3] 100/250 users done
[review3] 125/250 users done
[review3] 150/250 users done
[review3] 175/250 users done
[review3] 200/250 users done
[review3] 225/250 users done
[review3] 250/250 users done
wrote results/review3/review3_amazon_itemknn.json


CompletedProcess(args=['/Users/mlouhichi/Desktop/actionShap/.venv/bin/python', 'scripts/run_review3_experiments.py', '--dataset', 'amazon', '--users', '250', '--exact-users', '100', '--exact-max', '12'], returncode=0)

## 3. SASRec (competitive sequential recommender, torch)

`WeightedSASRec` computes the user representation at scoring time as a
history-weighted sum of encoded positions, so masking/bounded downweighting
changes scores without retraining - the interface the protocol requires.
Skipped automatically when torch is unavailable.

In [6]:
try:
    import torch
    for ds in ("movielens", "amazon"):
        subprocess.run([sys.executable, "scripts/run_review3_experiments.py",
                        "--dataset", ds, "--model", "sasrec", "--users", "200"],
                       check=True)
except Exception as e:
    print("skipping SASRec:", e)

/Users/mlouhichi/Desktop/actionShap/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


[review3] 25/200 users done
[review3] 50/200 users done
[review3] 75/200 users done
[review3] 100/200 users done
[review3] 125/200 users done
[review3] 150/200 users done
[review3] 175/200 users done
[review3] 200/200 users done
wrote results/review3/review3_movielens_sasrec.json


/Users/mlouhichi/Desktop/actionShap/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


[review3] 25/200 users done
[review3] 50/200 users done
[review3] 75/200 users done
[review3] 100/200 users done
[review3] 125/200 users done
[review3] 150/200 users done
[review3] 175/200 users done
[review3] 200/200 users done
wrote results/review3/review3_amazon_sasrec.json


## 4. Enlarged full-catalogue cohort (1,000 users)

Review issue 12: the archived full-catalogue NRegret active samples were
n=20/5. Rerun the schema-v2 pipeline with a 1,000-user full-catalogue subset
per dataset (ItemKNN). Afterwards regenerate assets with
`scripts/make_paper_assets.py --raw results/raw --out <paper-dir>`.

In [7]:
for ds_args in (["--dataset-format","ml1m","--dataset-path","data/ml-1m/ratings.dat",
                   "--dataset-name","MovieLens-1M"],
                  ["--dataset-format","csv","--dataset-path","data/amazon-digital-music/interactions.csv",
                   "--dataset-name","Amazon-Digital-Music"]):
    subprocess.run([sys.executable, "scripts/run_recommendation.py", *ds_args,
                    "--model","itemknn","--analysis-role","full_catalogue",
                    "--condition","full_catalogue","--full-catalog",
                    "--full-catalog-users","1000",
                    "--output", f"results/raw/{ds_args[3].split('/')[-1]}_fullcatalog1000.json"],
                   check=False)

usage: run_recommendation.py [-h] --ratings RATINGS
                             [--dataset-format {ml1m,csv}]
                             [--dataset-name DATASET_NAME]
                             [--user-column USER_COLUMN]
                             [--item-column ITEM_COLUMN]
                             [--timestamp-column TIMESTAMP_COLUMN]
                             [--rating-column RATING_COLUMN]
                             [--rating-threshold RATING_THRESHOLD]
                             [--minimum-interactions MINIMUM_INTERACTIONS]
                             [--output OUTPUT] [--gate-only]
                             [--analysis-role {primary,full_catalogue,sensitivity}]
                             [--condition CONDITION]
                             [--model {profile,itemknn}]
                             [--model-role {primary,robustness}]
                             [--n-max N_MAX] [--candidate-k EVALUATION_SIZE]
                             [--full-catalog] [--

## 5. Aggregate results and build Table D4

In [8]:
import json, statistics, csv
import pandas as pd
from pathlib import Path
out = Path("results/review3")
LABEL = {"review3_amazon_itemknn": "Amazon ItemKNN", "review3_amazon_sasrec": "Amazon SASRec",
         "review3_movielens_itemknn": "ML-1M ItemKNN", "review3_movielens_sasrec": "ML-1M SASRec"}
def ok(v):
    return v is not None and not (isinstance(v, float) and v != v)
rows = []
for f in sorted(out.glob("review3_*.json")):
    payload = json.loads(f.read_text())
    recs = payload["records"]
    def mean(key):
        vals = [r[key] for r in recs if ok(r.get(key))]
        return statistics.mean(vals) if vals else float("nan")
    exact = [r["exact_mc_error"] for r in recs if "exact_mc_error" in r]
    rows.append({
        "label": LABEL.get(f.stem, f.stem), "n": payload["n_users"],
        "exact_max_abs_err": statistics.mean([e["max_abs_error"] for e in exact]) if exact else float("nan"),
        "exact_rank_rho": statistics.mean([e["rank_spearman"] for e in exact if ok(e.get("rank_spearman"))]) if exact else float("nan"),
        "interaction_ratio": statistics.mean([r["interaction_summary"]["interaction_to_singleton_ratio"] for r in recs]),
        "additive_vs_realized_rho": statistics.mean([r["additive_vs_realized"]["spearman_additive_realized"] for r in recs if ok(r["additive_vs_realized"].get("spearman_additive_realized"))]),
        "AIA_shapley": mean("aia_shapley"), "AIA_lime_binary": mean("aia_binary_lime"),
        "AIA_lime_bounded": mean("aia_bounded_lime_bin"), "AIA_lime_continuous": mean("aia_bounded_lime_cont"),
        "AIA_finite_diff": mean("aia_finite_diff"), "AIA_ig": mean("aia_ig"),
        "prospective_share": statistics.mean([r["prospective"]["target_is_generated_top1"] for r in recs]),
        "AIA_prospective": statistics.mean([r["prospective"]["aia_shapley_prospective"] for r in recs if ok(r["prospective"].get("aia_shapley_prospective"))]),
    })
df = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(df.round(3).to_string(index=False))   # no tabulate dependency
df.to_csv(out / "summary.csv", index=False)


         label   n  exact_max_abs_err  exact_rank_rho  interaction_ratio  additive_vs_realized_rho  AIA_shapley  AIA_lime_binary  AIA_lime_bounded  AIA_lime_continuous  AIA_finite_diff  AIA_ig  prospective_share  AIA_prospective
Amazon ItemKNN 250              0.000           0.783              0.078                     0.842        0.669            0.739             0.738                0.734            1.000   0.967              0.992            0.727
 Amazon SASRec 200              0.013           0.395              0.782                     0.453        0.140            0.760             0.942                0.827            0.734   0.722              0.995            0.798
 ML-1M ItemKNN 250              0.002           0.982              0.071                     0.905        0.697            0.905             0.936                0.938            0.958   0.953              0.908            0.796
  ML-1M SASRec 200              0.018           0.688              0.206            

In [9]:
from pathlib import Path
TABLES = CODE.parent / "actionshap-overleaf" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
def fmt(v, w=3):
    return "--" if v != v else f"{v:.{w}f}"
lines = ["% Table D4 (generated by ActionShap_Review3_Experiments.ipynb).",
"\\begin{table}[t]\\centering\\scriptsize\\setlength{\\tabcolsep}{2.5pt}",
"\\caption{Review-3 replication aggregates per run: exact-Shapley validation of the",
"MC estimator (max absolute error and rank $\\rho$ on the enumerated subset),",
"interaction strength relative to singleton effects, additive-vs-realized pair",
"ranking, and bounded/intervention-aware baseline alignment; last column is the",
"prospective (non-target-conditioned) Shapley alignment. $--$ = undefined",
"(constant exact spectrum or constant effect vectors).}",
"\\label{tab:review3}",
"\\begin{tabular}{@{}lrrrrrrr@{}}",
"\\toprule",
"Run & exact err & exact $\\rho$ & interact. & add./real. & Shap. & LIME bin. & prosp.\\ AIA \\\\",
"\\midrule"]
for r in rows:
    lines.append(f"{r['label']} & {fmt(r['exact_max_abs_err'],4)} & {fmt(r['exact_rank_rho'])} & "
                 f"{fmt(r['interaction_ratio'])} & {fmt(r['additive_vs_realized_rho'])} & "
                 f"{fmt(r['AIA_shapley'])} & {fmt(r['AIA_lime_bounded'])} & {fmt(r['AIA_prospective'])} \\\\")
lines += ["\\bottomrule","\\end{tabular}","\\end{table}"]
(TABLES / "review3_tableD4.tex").write_text("\n".join(lines)+"\n")
(out / "summary.md").write_text(
    "# Review-3 replication summary\n\n```\n" + df.round(3).to_string(index=False) + "\n```\n")
print("wrote", TABLES / "review3_tableD4.tex", "and", out / "summary.md")


wrote /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/actionshap-overleaf/tables/review3_tableD4.tex and results/review3/summary.md


In [10]:
# Extension tables (review-3 high items): equal-budget curves, rho-response,
# kappa sensitivity, unconditional regret, per-method prospective AIA.
ext_rows, curve_rows = [], []
for f in sorted(out.glob("review3_*.json")):
    d = json.loads(f.read_text())
    run = LABEL.get(f.stem, f.stem)
    recs = d["records"]
    un = [r["unconditional"]["unconditional_regret"] for r in recs if "unconditional" in r]
    pm = {}
    for r in recs:
        for k, v in r.get("prospective", {}).items():
            if k.startswith("aia_") and ok(v): pm.setdefault(k, []).append(v)
    ext_rows.append((run, statistics.mean(un) if un else float("nan"),
                     {k: statistics.mean(v) for k, v in pm.items()}))
    for c in d.get("curves", {}).get("equal_budget", []):
        curve_rows.append((run, "budget", f"{c['method']} B={c['budget']}", c["mean_bounded_aia"]))
    for c in d.get("curves", {}).get("rho_curve", []):
        curve_rows.append((run, "rho", f"rho={c['rho']}", c["mean_bounded_aia"]))
    for c in d.get("curves", {}).get("kappa_curve", []):
        curve_rows.append((run, "kappa", f"kappa={c['kappa']}", c["mean_bounded_aia"]))
lines = ["% Extension tables (generated).",
"\\begin{table}[t]\\centering\\scriptsize\\setlength{\\tabcolsep}{2.5pt}",
"\\caption{Review-3 extensions: unconditional (full-cohort) regret and",
"prospective bounded AIA per explainer (non-target-conditioned audits).}",
"\\label{tab:review3-ext}",
"\\begin{tabular}{@{}lrrrrrr@{}}",
"\\toprule",
"Run & uncond.\\ regret & prosp.\\ Shap. & prosp.\\ LIME b. & prosp.\\ bLIME & prosp.\\ FD & prosp.\\ IG \\\\",
"\\midrule"]
for run, un, pm in ext_rows:
    lines.append(f"{run} & {fmt(un)} & {fmt(pm.get('aia_shapley_prospective'))} & "
                 f"{fmt(pm.get('aia_lime_binary_prospective'))} & {fmt(pm.get('aia_bounded_lime_prospective'))} & "
                 f"{fmt(pm.get('aia_finite_diff_prospective'))} & {fmt(pm.get('aia_ig_prospective'))} \\\\")
lines += ["\\bottomrule", "\\end{tabular}",
"\\par\\vspace{2mm}",
"\\begin{tabular}{@{}lllr@{}}",
"\\toprule",
"Run & Curve & Point & Bounded AIA \\",
"\\midrule"]
for run, kind, point, val in curve_rows:
    lines.append(f"{run} & {kind} & {point} & {val:.3f} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
(TABLES / "review3_extensions.tex").write_text("\n".join(lines) + "\n")
print("wrote", TABLES / "review3_extensions.tex")


wrote /Users/mlouhichi/Desktop/actionShap/next-paper/paper-ideas/ActionShap/actionshap-overleaf/tables/review3_extensions.tex


## 6. Verify and push

1. `pytest -q` stays green.
2. Inspect `results/review3/summary.md`.
3. Commit and push this branch:

```
git add paper-ideas/ActionShap/code/results/review3 paper-ideas/ActionShap/code/tables/review3_tableD4.tex
git commit -m "ActionShap: review-3 replication results"
git push origin <branch>
```

The manuscript integration (Table D4 into Appendix D, exact-Shapley validation
sentence in §6.5, runtime numbers in §6.6, SASRec row in Table 2 if it passes
the masking gate and exceeds popularity) is then performed from these
artifacts.